# fase_3 - script_afrida Migration

This notebook handles migration of database from old DB to new DB for fase 3.

**Purpose**: Benerin database lama ke database baru untuk bagian [NAMA TABEL]

In [1]:
import sys
import os
import mysql.connector
import pandas as pd
sys.path.append(os.path.abspath('..'))
from config import get_db_config
import warnings
warnings.filterwarnings('ignore')

## 1. Connect ke Database

In [2]:
# Connect ke database config
config = get_db_config()
# Ambil host dari salah satu config (misal db_old)
print(f'Database config loaded: {config["db_old"]["host"]}')

# Connect ke DB Lama
db_old = mysql.connector.connect(**config['db_old'])
cursor_old = db_old.cursor(dictionary=True)
print(f'Connected to old database: {config["db_old"]["database"]}')

# Connect ke DB Baru
db_new = mysql.connector.connect(**config['db_new'])
cursor_new = db_new.cursor(dictionary=True)
print(f'Connected to new database: {config["db_new"]["database"]}')

db_future = mysql.connector.connect(**config['db_future'])
cursor_future = db_future.cursor(dictionary=True)
print(f'Connected to future database: {config["db_future"]["database"]}')


Database config loaded: localhost
Connected to old database: dataleap_juni
Connected to new database: dataleap_v5_migration
Connected to future database: db_future


## 2. Surat Keluar dan Verifikasi Surat Keluar

In [3]:
# Ambil data suratkeluar
df_sk_raw = pd.read_sql("SELECT * FROM suratkeluar", db_old)
print("=== SURAT KELUAR (lama) ===")
print("Jumlah:", len(df_sk_raw))
display(df_sk_raw.head(5))
print("\nKolom:", df_sk_raw.columns.tolist())
print("\nTipe data:")
print(df_sk_raw.dtypes)
print("\nMissing values:")
print(df_sk_raw.isnull().sum())

# Cek nilai unik kolom status (untuk enum)
print("\nNilai unik 'status':", df_sk_raw['status'].unique())

# Cek apakah idsurat unik?
print("\nApakah idsurat unik?", df_sk_raw['idsurat'].is_unique)

=== SURAT KELUAR (lama) ===
Jumlah: 231


,idsurat,keterangan,link,idusers,status,created_at,nosurat,catatan
0,29,Penawaran Pelatihan Business English ke PT. La...,https://docs.google.com/document/d/1wBK62noEJw...,U00011,Disetujui,2023-11-13 15:41:58,105/LEAP/BD/XI/2023,
1,27,PERJANJIAN KERJASAMA / MEMORANDUM OF UNDERSTAN...,https://docs.google.com/document/d/1Edeb7hWaBq...,U00011,Disetujui,2023-10-26 17:37:59,102/LEAP/BD/X/2023,
2,28,Sertifikat / Sertifikat Kelas APK Private / 1 ...,https://drive.google.com/drive/folders/1JMrPJF...,U00026,Disetujui,2023-11-10 16:10:04,Sertif 002a/APEX/XI/2324/02 (Page 1) dan 002b/...,
3,26,Beasiswa Siswa GE,https://docs.google.com/document/d/1jMTK_3b57e...,U00011,Disetujui,2023-10-25 16:57:38,101/LEAP/BD/X/2023,
4,24,Surat Permohonan Uji Kompetensi dan Penggunaan...,https://docs.google.com/document/d/1xyhc4T-FlR...,U00011,Direvisi,2023-09-11 17:57:49,090A/LEAP/IX/2023- 090B/LEAP/IX/2023,Lengkapi Tanggal Pelaksanaan dengan tanggal ya...



Kolom: ['idsurat', 'keterangan', 'link', 'idusers', 'status', 'created_at', 'nosurat', 'catatan']

Tipe data:
idsurat                int64
keterangan            object
link                  object
idusers               object
status                object
created_at    datetime64[ns]
nosurat               object
catatan               object
dtype: object

Missing values:
idsurat       0
keterangan    0
link          0
idusers       0
status        0
created_at    0
nosurat       2
catatan       2
dtype: int64

Nilai unik 'status': ['Disetujui' 'Direvisi' 'Ditolak' 'Revisi' 'Diajukan']

Apakah idsurat unik? True


In [4]:
# Mapping status
status_mapping = {
    'Direvisi': 'Sudah Revisi',
    'Disetujui': 'Disetujui',
    'Ditolak': 'Ditolak'
}

# Bentuk DataFrame final
df_sk = pd.DataFrame({
    'id_sk': df_sk_raw['idsurat'].astype('Int64'),
    'id_user': df_sk_raw['idusers'].str.strip(),
    'keterangan_sk': df_sk_raw['keterangan'].str.strip(),
    'link_dokumen_sk': df_sk_raw['link'].str.strip(),
    'status_sk': df_sk_raw['status'].map(status_mapping),
    'nomor_sk': df_sk_raw['nosurat'].str.strip(),
    'catatan_sk': df_sk_raw['catatan'].fillna('').str.strip(),
    'created_at': pd.to_datetime(df_sk_raw['created_at'])
})

# Simpan mapping untuk histori
mapping_sk = dict(zip(df_sk_raw['idsurat'], df_sk['id_sk']))

In [5]:
display(df_sk.head())

,id_sk,id_user,keterangan_sk,link_dokumen_sk,status_sk,nomor_sk,catatan_sk,created_at
0,29,U00011,Penawaran Pelatihan Business English ke PT. La...,https://docs.google.com/document/d/1wBK62noEJw...,Disetujui,105/LEAP/BD/XI/2023,,2023-11-13 15:41:58
1,27,U00011,PERJANJIAN KERJASAMA / MEMORANDUM OF UNDERSTAN...,https://docs.google.com/document/d/1Edeb7hWaBq...,Disetujui,102/LEAP/BD/X/2023,,2023-10-26 17:37:59
2,28,U00026,Sertifikat / Sertifikat Kelas APK Private / 1 ...,https://drive.google.com/drive/folders/1JMrPJF...,Disetujui,Sertif 002a/APEX/XI/2324/02 (Page 1) dan 002b/...,,2023-11-10 16:10:04
3,26,U00011,Beasiswa Siswa GE,https://docs.google.com/document/d/1jMTK_3b57e...,Disetujui,101/LEAP/BD/X/2023,,2023-10-25 16:57:38
4,24,U00011,Surat Permohonan Uji Kompetensi dan Penggunaan...,https://docs.google.com/document/d/1xyhc4T-FlR...,Sudah Revisi,090A/LEAP/IX/2023- 090B/LEAP/IX/2023,Lengkapi Tanggal Pelaksanaan dengan tanggal ya...,2023-09-11 17:57:49


In [6]:
# Cek tiap kolom
kolom_lama = ['idsurat', 'keterangan', 'link', 'idusers', 'status', 'created_at', 'nosurat', 'catatan']
kolom_baru = ['id_sk', 'id_user', 'keterangan_sk', 'link_dokumen_sk', 'status_sk', 'nomor_sk', 'catatan_sk', 'created_at']

print("=== PENGECEKAN KOLOM SURAT KELUAR ===\n")

for lama, baru in zip(kolom_lama, kolom_baru):
    print(f"--- {lama} → {baru} ---")
    print(f"  Missing: {df_sk_raw[lama].isnull().sum()}")
    if df_sk_raw[lama].dtype == 'object':
        unik = df_sk_raw[lama].dropna().unique()
        print(f"  Unique ({len(unik)}): {unik[:20]}...")  # maks 20
    else:
        print(f"  Min: {df_sk_raw[lama].min()} | Max: {df_sk_raw[lama].max()}")
    print()

=== PENGECEKAN KOLOM SURAT KELUAR ===

--- idsurat → id_sk ---
  Missing: 0
  Min: 24 | Max: 272

--- keterangan → id_user ---
  Missing: 0
  Unique (224): ['Penawaran Pelatihan Business English ke PT. Lautan Natural Krimerindo'
 'PERJANJIAN KERJASAMA / MEMORANDUM OF UNDERSTANDING (MoU) IN-HOUSE TRAINING : Training Dasar Editing Capcut  CV. Rabbani (Fashion Retail)'
 'Sertifikat / Sertifikat Kelas APK Private / 1 Peserta / No. Sertif 002a/APEX/XI/2324/02 (Page 1)  dan 002b/APEX/XI/2324/02 (Page 2) '
 'Beasiswa Siswa GE'
 'Surat Permohonan Uji Kompetensi dan Penggunaan Tempat Sebagai TUK'
 'Surat Pengantar Kajian Mitra Prakerja'
 'Konfirmasi Penerimaan Permohonan Data dan Koordinasi Pengukuran Kebutuhan Talenta Digital Indonesia berdasarkan permintaan dari KOMINFO'
 'Surat Rekomendasi CV.Rabbani' 'TOR LeapXperience Holiday Program'
 'Surat Undangan ke Sekolah LeapXperience Holiday Program'
 'Sertifikat In-house training capcut Leap x CV Rabbani '
 'NDA LMS Studiokerja (PT. EKI)' 'ToR Gr

In [7]:
pd.read_sql("DESCRIBE surat_keluar", db_new)

,Field,Type,Null,Key,Default,Extra
0,id_sk,bigint(20) unsigned,NO,PRI,None,auto_increment
1,id_user,varchar(15),YES,MUL,None,
2,keterangan_sk,text,NO,,None,
3,link_dokumen_sk,varchar(255),NO,,None,
4,status_sk,"enum('Diajukan','Sudah Revisi','Disetujui','Di...",NO,,Diajukan,
5,nomor_sk,varchar(255),YES,,None,
6,catatan_sk,text,NO,,None,
7,created_at,timestamp,NO,,current_timestamp(),


In [8]:
print("=== SHOW COLUMNS FROM Surat_Keluar ===")
display(pd.read_sql("SHOW COLUMNS FROM surat_keluar", db_future))

=== SHOW COLUMNS FROM Surat_Keluar ===


,Field,Type,Null,Key,Default,Extra
0,id_sk,bigint(20) unsigned,NO,PRI,None,auto_increment
1,id_user,varchar(15),YES,MUL,None,
2,keterangan_sk,text,NO,,None,
3,link_dokumen_sk,varchar(255),NO,,None,
4,status_sk,"enum('Diajukan','Sudah Revisi','Disetujui','Di...",NO,,Diajukan,
5,nomor_sk,varchar(100),NO,,None,
6,catatan_sk,text,NO,,None,
7,created_at,timestamp,NO,,current_timestamp(),


In [9]:
print("Nilai unik 'status' di data lama:")
print(df_sk_raw['status'].value_counts())

Nilai unik 'status' di data lama:
status
Disetujui    226
Diajukan       2
Direvisi       1
Ditolak        1
Revisi         1
Name: count, dtype: int64


In [10]:
# =========================================================
# CLEANING KHUSUS UNTUK SURAT KELUAR (df_sk)
# =========================================================
print("Membersihkan kolom nomor_sk dan catatan_sk...")

# Ganti None/nan atau string kosong dengan teks default
df_sk['nomor_sk'] = df_sk['nomor_sk'].fillna('Nomor surat belum diisi')
df_sk['nomor_sk'] = df_sk['nomor_sk'].replace('', 'Nomor surat belum diisi')

df_sk['catatan_sk'] = df_sk['catatan_sk'].fillna('Tidak ada catatan')
df_sk['catatan_sk'] = df_sk['catatan_sk'].replace('', 'Tidak ada catatan')

# Cek hasil (tampilkan beberapa baris yang sebelumnya kosong)
print("Contoh setelah pengisian:")
display(df_sk[['nomor_sk', 'catatan_sk']].head(10))

Membersihkan kolom nomor_sk dan catatan_sk...
Contoh setelah pengisian:


,nomor_sk,catatan_sk
0,105/LEAP/BD/XI/2023,Tidak ada catatan
1,102/LEAP/BD/X/2023,Tidak ada catatan
2,Sertif 002a/APEX/XI/2324/02 (Page 1) dan 002b/...,Tidak ada catatan
3,101/LEAP/BD/X/2023,Tidak ada catatan
4,090A/LEAP/IX/2023- 090B/LEAP/IX/2023,Lengkapi Tanggal Pelaksanaan dengan tanggal ya...
5,106/LEAP/XI/2023,Tidak ada catatan
6,107/LEAP/XI/2023,Tidak ada catatan
7,103/LEAP/XI/2023,Tidak ada catatan
8,108/LEAP/BD/XI/2023,Tidak ada catatan
9,109/LEAP/BD/XI/2023,Tidak ada catatan


In [11]:
# Ambil data suratkeluar_histori
df_hist_raw = pd.read_sql("SELECT * FROM suratkeluar_histori", db_old)
print("\n=== SURAT KELUAR HISTORI (lama) ===")
print("Jumlah:", len(df_hist_raw))
display(df_hist_raw.head(10))
print("\nKolom:", df_hist_raw.columns.tolist())
print("\nTipe data:")
print(df_hist_raw.dtypes)
print("\nMissing values:")
print(df_hist_raw.isnull().sum())

# Cek nilai unik status histori
print("\nNilai unik 'status':", df_hist_raw['status'].unique())


=== SURAT KELUAR HISTORI (lama) ===
Jumlah: 513


,idstatus,idsurat,status,catatan,created_at
0,7,12,Diajukan,None,2023-06-12 13:32:50
1,8,13,Diajukan,None,2023-06-12 13:33:07
2,9,14,Diajukan,None,2023-06-12 14:28:56
3,10,15,Diajukan,None,2023-06-30 15:30:35
4,11,16,Diajukan,None,2023-07-01 20:35:43
5,12,17,Diajukan,None,2023-07-03 15:00:16
6,13,18,Diajukan,None,2023-07-03 15:16:42
7,14,19,Diajukan,None,2023-07-03 15:57:42
8,15,20,Diajukan,None,2023-07-03 15:58:54
9,16,21,Diajukan,None,2023-07-03 16:00:00



Kolom: ['idstatus', 'idsurat', 'status', 'catatan', 'created_at']

Tipe data:
idstatus               int64
idsurat                int64
status                object
catatan               object
created_at    datetime64[ns]
dtype: object

Missing values:
idstatus        0
idsurat         0
status          0
catatan       265
created_at      0
dtype: int64

Nilai unik 'status': ['Diajukan' 'Revisi' 'Disetujui' 'Direvisi' '' 'Ditolak']


In [12]:
pd.read_sql("DESCRIBE verifikasi_surat_keluar", db_new)

,Field,Type,Null,Key,Default,Extra
0,id_verifikasi_surat,bigint(20) unsigned,NO,PRI,None,auto_increment
1,id_sk,bigint(20) unsigned,YES,MUL,None,
2,status_verifikasi_sk,"enum('Revisi','Diajukan','Disetujui','Ditolak')",NO,,Diajukan,
3,catatan_verifikasi_sk,text,YES,,None,
4,created_at,timestamp,NO,,current_timestamp(),


In [13]:
missing_sk = df_hist_raw[~df_hist_raw['idsurat'].isin(df_sk_raw['idsurat'])]
print(f"idsurat di histori yang tidak ada di surat_keluar: {len(missing_sk)}")
if len(missing_sk) > 0:
    print(missing_sk[['idstatus', 'idsurat']].head())

idsurat di histori yang tidak ada di surat_keluar: 36
   idstatus  idsurat
0         7       12
1         8       13
2         9       14
3        10       15
4        11       16


In [14]:
try:
    cursor_new.execute("ALTER TABLE verifikasi_surat_keluar MODIFY COLUMN catatan_verifikasi_sk text NULL;")
    db_new.commit()
    print("✅ Kolom catatan_verifikasi_sk sekarang NULLABLE.")
except Exception as e:
    print("Gagal:", e)

✅ Kolom catatan_verifikasi_sk sekarang NULLABLE.


In [15]:
try:
    cursor_new.execute("""
        ALTER TABLE verifikasi_surat_keluar 
          MODIFY COLUMN catatan_verifikasi_sk text NULL,
          MODIFY COLUMN status_verifikasi_sk enum('Revisi','Diajukan','Disetujui','Ditolak') NOT NULL DEFAULT 'Diajukan';
    """)
    db_new.commit()
    print("✅ Tabel berhasil diubah.")
except Exception as e:
    print("Gagal:", e)

✅ Tabel berhasil diubah.


In [16]:
# 1. Copy data mentah histori
df_verif = df_hist_raw.copy()

# 2. Set id_sk: ambil dari idsurat jika ada di surat_keluar, jika tidak set None (NULL di DB)
#    Gunakan tipe Int64 (Nullable Integer) agar sinkron dengan bigint(20) unsigned
id_surat_valid = set(df_sk_raw['idsurat'])
df_verif['id_sk'] = df_verif['idsurat'].apply(lambda x: x if x in id_surat_valid else None).astype('Int64')

# 3. Mapping status lama ke enum baru
#    Enum baru: 'Revisi','Diajukan','Disetujui','Ditolak'
mapping_status_verif = {
    'Diajukan': 'Diajukan',
    'Revisi': 'Revisi',
    'Disetujui': 'Disetujui',
    'Direvisi': 'Revisi',   # kita setarakan dengan Revisi
    'Ditolak': 'Ditolak',
    '': 'Diajukan'          # string kosong dianggap pengajuan baru
}
df_verif['status_verifikasi_sk'] = df_verif['status'].map(mapping_status_verif).fillna('Diajukan')

# 4. catatan_verifikasi_sk: isi None jika missing/kosong agar menjadi NULL di database
df_verif['catatan_verifikasi_sk'] = df_verif['catatan'].apply(lambda x: str(x).strip() if pd.notna(x) and str(x).strip() != '' else None)

# 5. created_at tetap sebagai datetime
df_verif['created_at'] = pd.to_datetime(df_verif['created_at'])

# 6. Pilih hanya kolom yang diperlukan (tanpa id_verifikasi_surat, biar auto_increment)
df_verif_final = df_verif[['id_sk', 'status_verifikasi_sk', 'catatan_verifikasi_sk', 'created_at']]

# 7. Cek hasil final tipe data
print("✅ verifikasi_surat_keluar final:")
display(df_verif_final.head(10))
print("\nTipe data (Aman untuk Migration):")
print(df_verif_final.dtypes)
print("\nMissing values (id_sk & catatan boleh NULL):")
print(df_verif_final.isnull().sum())

✅ verifikasi_surat_keluar final:


,id_sk,status_verifikasi_sk,catatan_verifikasi_sk,created_at
0,<NA>,Diajukan,None,2023-06-12 13:32:50
1,<NA>,Diajukan,None,2023-06-12 13:33:07
2,<NA>,Diajukan,None,2023-06-12 14:28:56
3,<NA>,Diajukan,None,2023-06-30 15:30:35
4,<NA>,Diajukan,None,2023-07-01 20:35:43
5,<NA>,Diajukan,None,2023-07-03 15:00:16
6,<NA>,Diajukan,None,2023-07-03 15:16:42
7,<NA>,Diajukan,None,2023-07-03 15:57:42
8,<NA>,Diajukan,None,2023-07-03 15:58:54
9,<NA>,Diajukan,None,2023-07-03 16:00:00



Tipe data (Aman untuk Migration):
id_sk                             Int64
status_verifikasi_sk             object
catatan_verifikasi_sk            object
created_at               datetime64[ns]
dtype: object

Missing values (id_sk & catatan boleh NULL):
id_sk                     36
status_verifikasi_sk       0
catatan_verifikasi_sk    511
created_at                 0
dtype: int64


In [17]:
pd.read_sql("DESCRIBE verifikasi_surat_keluar", db_future)

,Field,Type,Null,Key,Default,Extra
0,id_verifikasi_surat,bigint(20) unsigned,NO,PRI,None,auto_increment
1,id_sk,bigint(20) unsigned,YES,MUL,None,
2,status_verifikasi_sk,"enum('Revisi','Diterima','Disetujui','Ditolak')",NO,,Diterima,
3,catatan_verifikasi_sk,text,NO,,None,
4,created_at,timestamp,NO,,current_timestamp(),


In [18]:
# Ubah id_sk ke nullable integer (Int64) agar cocok dengan bigint unsigned NULL
df_verif_final['id_sk'] = df_verif_final['id_sk'].astype('Int64')

# Cek tipe data akhir
print("Tipe data setelah penyesuaian:")
print(df_verif_final.dtypes)

Tipe data setelah penyesuaian:
id_sk                             Int64
status_verifikasi_sk             object
catatan_verifikasi_sk            object
created_at               datetime64[ns]
dtype: object


In [19]:
# =========================================================
# CLEANING UNTUK verif_final: catatan_verifikasi_sk
# =========================================================
print("Membersihkan kolom catatan_verifikasi_sk...")

# Ganti None/nan atau string kosong dengan 'Tidak ada catatan'
df_verif_final['catatan_verifikasi_sk'] = df_verif_final['catatan_verifikasi_sk'].fillna('Tidak ada catatan')
df_verif_final['catatan_verifikasi_sk'] = df_verif_final['catatan_verifikasi_sk'].replace('', 'Tidak ada catatan')

# Cek hasil
print("Contoh setelah pengisian:")
display(df_verif_final.head(10))

Membersihkan kolom catatan_verifikasi_sk...
Contoh setelah pengisian:


,id_sk,status_verifikasi_sk,catatan_verifikasi_sk,created_at
0,<NA>,Diajukan,Tidak ada catatan,2023-06-12 13:32:50
1,<NA>,Diajukan,Tidak ada catatan,2023-06-12 13:33:07
2,<NA>,Diajukan,Tidak ada catatan,2023-06-12 14:28:56
3,<NA>,Diajukan,Tidak ada catatan,2023-06-30 15:30:35
4,<NA>,Diajukan,Tidak ada catatan,2023-07-01 20:35:43
5,<NA>,Diajukan,Tidak ada catatan,2023-07-03 15:00:16
6,<NA>,Diajukan,Tidak ada catatan,2023-07-03 15:16:42
7,<NA>,Diajukan,Tidak ada catatan,2023-07-03 15:57:42
8,<NA>,Diajukan,Tidak ada catatan,2023-07-03 15:58:54
9,<NA>,Diajukan,Tidak ada catatan,2023-07-03 16:00:00


## 3. Surat Tugas dan Surat Tugas Anggota

In [20]:
# Data surattugas dari DB lama
df_st_raw = pd.read_sql("SELECT * FROM surattugas", db_old)
print("=== SURAT TUGAS (lama) ===")
print("Jumlah:", len(df_st_raw))
display(df_st_raw.head(5))
print("\nKolom:", df_st_raw.columns.tolist())
print("\nTipe data:")
print(df_st_raw.dtypes)
print("\nMissing values:")
print(df_st_raw.isnull().sum())
print("\nNilai unik 'status':", df_st_raw['status'].unique())
print("Nilai unik 'jenis':", df_st_raw['jenis'].unique())

=== SURAT TUGAS (lama) ===
Jumlah: 139


,idsurat,acara,undangan,waktu,lokasi,jenis,status,idusers,created_at,nosurat,catatan,link,linklaporan,notelaporan,ket,notebatal
0,24,Sosialisasi Kerjasama LKP dan PKBM dengan Seme...,Dinas Pendidikan Kota Surabaya (bu Hilda),"<table class=""NormalTable"">\r\n<tbody>\r\n<tr>...","Tempat : Ruang Bung Tomo, Dinas Pendidikan Kot...",Offline,Disetujui,U00015,2023-09-19 13:55:50,027/LEAP/ST/IX/2023,,https://docs.google.com/document/d/1LBx4WpqXcY...,https://docs.google.com/document/d/1jJdGMkonp_...,None,None,None
1,23,Pertemuan Rutin Larasdikdudi bulan September 2023,https://drive.google.com/drive/u/3/folders/1OI...,"<p><span class=""fontstyle0"">Hari, Tanggal : Ra...",AULA SMK Negeri 2 Surabaya Jalan Tentara Genie...,Offline,Disetujui,U00015,2023-09-19 13:54:26,028/LEAP/ST/IX/2023,,https://docs.google.com/document/d/1wMOD-N6oMQ...,https://docs.google.com/document/d/1xMv7AK2L9S...,done,None,None
2,22,TEDxSurabaya Translators:\r\n\r\n1. Simulasi C...,TEDxSurabaya,<p>1. Simulasi Coaching:&nbsp;15 September 202...,Topic: Connect to TEDxSurabaya Join Zoom Meeti...,Online,Disetujui,U00016,2023-09-13 13:01:52,025/LEAP/ST/VIII/2023,,https://docs.google.com/document/d/10Zi6g0R9RR...,https://docs.google.com/document/d/1pa9G3E3eUa...,laporan kegiatan done,None,None
3,20,Pertemuan rutin di bulan Juli 2023 forum Laras...,Larasdikdudi,"<p><span class=""fontstyle0"">Hari, Tanggal : Ka...",SMK Teknik PAL Surabaya Jalan Ujung Surabaya,Offline,Disetujui,U00015,2023-07-25 09:56:19,023/LEAP/ST/VII/2023,,https://docs.google.com/document/d/1rILu6FE8Bd...,https://docs.google.com/document/d/1IAhWT4XjbX...,done/laporan sudah diisi,None,None
4,21,Temu Warga RT 01 RW 08,Pengurus RT 01/ RW08,"<p>Rabu, 23 Agustus 2023<br />jam 19.00-selesa...",Balai RW,Offline,Disetujui,U00015,2023-08-22 17:43:48,024/LEAP/ST/VIII/2023,,https://docs.google.com/document/d/1OQxpdKYfDF...,https://docs.google.com/document/d/1XrvRmYckBw...,None,None,None



Kolom: ['idsurat', 'acara', 'undangan', 'waktu', 'lokasi', 'jenis', 'status', 'idusers', 'created_at', 'nosurat', 'catatan', 'link', 'linklaporan', 'notelaporan', 'ket', 'notebatal']

Tipe data:
idsurat                 int64
acara                  object
undangan               object
waktu                  object
lokasi                 object
jenis                  object
status                 object
idusers                object
created_at     datetime64[ns]
nosurat                object
catatan                object
link                   object
linklaporan            object
notelaporan            object
ket                    object
notebatal              object
dtype: object

Missing values:
idsurat          0
acara            0
undangan         0
waktu            0
lokasi           0
jenis            0
status           0
idusers          0
created_at       0
nosurat          0
catatan          0
link             0
linklaporan      0
notelaporan     59
ket            136
notebata

In [21]:
# Struktur tabel surat_tugas di DB baru
print("\nStruktur surat_tugas (db_old):")
pd.read_sql("DESCRIBE surattugas", db_old)


Struktur surat_tugas (db_old):


,Field,Type,Null,Key,Default,Extra
0,idsurat,int(11),NO,PRI,None,auto_increment
1,acara,text,NO,,None,
2,undangan,text,NO,,'',
3,waktu,text,NO,,'',
4,lokasi,text,NO,,'',
5,jenis,varchar(100),NO,,,
6,status,varchar(100),NO,,,
7,idusers,varchar(6),NO,MUL,,
8,created_at,datetime,NO,,current_timestamp(),
9,nosurat,varchar(50),YES,,None,


In [22]:
print("\nStruktur surat_tugas (db_future):")
pd.read_sql("DESCRIBE surat_tugas", db_future)


Struktur surat_tugas (db_future):


,Field,Type,Null,Key,Default,Extra
0,id_st,bigint(20) unsigned,NO,PRI,None,auto_increment
1,id_user,varchar(15),YES,MUL,None,
2,acara,varchar(255),NO,,None,
3,undangan,varchar(255),NO,,None,
4,waktu_acara,timestamp,NO,,None,
5,lokasi_acara,varchar(255),NO,,None,
6,jenis_kegiatan,"enum('Offline','Online')",NO,,None,
7,status_st,"enum('Diajukan','Disetujui','Revisi','Dibatalk...",YES,,Diajukan,
8,periode,varchar(100),YES,,None,
9,nomor_st,varchar(100),NO,,None,


In [23]:
# Lihat beberapa baris dan nilai unik
print(df_st_raw[['jenis', 'status']].head(5))
print("\nNilai unik jenis:", df_st_raw['jenis'].unique())
print("Nilai unik status:", df_st_raw['status'].unique())

     jenis     status
0  Offline  Disetujui
1  Offline  Disetujui
2   Online  Disetujui
3  Offline  Disetujui
4  Offline  Disetujui

Nilai unik jenis: ['Offline' 'Online']
Nilai unik status: ['Disetujui' 'Dibatalkan']


In [24]:
# 1. Konversi waktu (teks) ke datetime
df_st_raw['waktu_dt'] = pd.to_datetime(df_st_raw['waktu'], errors='coerce')

# 2. Bentuk DataFrame final DENGAN KOLOM 'periode'
df_st = pd.DataFrame({
    'id_st': df_st_raw['idsurat'],                # ID lama (sesuai kode sebelumnya)
    'id_user': df_st_raw['idusers'].str.strip(),
    'acara': df_st_raw['acara'].str.strip(),
    'undangan': df_st_raw['undangan'].str.strip(),
    'waktu_acara': df_st_raw['waktu_dt'],
    'lokasi_acara': df_st_raw['lokasi'].str.strip(),
    'jenis_kegiatan': df_st_raw['jenis'].str.strip(),
    'status_st': df_st_raw['status'].str.strip(),
    'periode': None,   # 🔥 TAMBAHKAN: NULL karena db_future nullable dan tidak ada sumber data
    'nomor_st': df_st_raw['nosurat'].fillna('').str.strip(),
    'catatan_st': df_st_raw['catatan'].fillna('').str.strip(),
    'link_st': df_st_raw['link'].fillna('').str.strip(),
    'link_laporan': df_st_raw['linklaporan'].fillna('').str.strip(),
    'catatan_laporan': df_st_raw['notelaporan'].fillna('').str.strip(),
    'keterangan_st': df_st_raw['ket'].fillna('').str.strip(),
    'catatan_pembatalan': df_st_raw['notebatal'].fillna('').str.strip(),
    'created_at': pd.to_datetime(df_st_raw['created_at'])
})

# 3. Cek hasil
print("✅ df_st siap dengan kolom periode = None")
display(df_st.head())
print("Tipe data:")
print(df_st.dtypes)
print("Missing values (periode):", df_st['periode'].isnull().sum())

✅ df_st siap dengan kolom periode = None


,id_st,id_user,acara,undangan,waktu_acara,lokasi_acara,jenis_kegiatan,status_st,periode,nomor_st,catatan_st,link_st,link_laporan,catatan_laporan,keterangan_st,catatan_pembatalan,created_at
0,24,U00015,Sosialisasi Kerjasama LKP dan PKBM dengan Seme...,Dinas Pendidikan Kota Surabaya (bu Hilda),NaT,"Tempat : Ruang Bung Tomo, Dinas Pendidikan Kot...",Offline,Disetujui,None,027/LEAP/ST/IX/2023,,https://docs.google.com/document/d/1LBx4WpqXcY...,https://docs.google.com/document/d/1jJdGMkonp_...,,,,2023-09-19 13:55:50
1,23,U00015,Pertemuan Rutin Larasdikdudi bulan September 2023,https://drive.google.com/drive/u/3/folders/1OI...,NaT,AULA SMK Negeri 2 Surabaya Jalan Tentara Genie...,Offline,Disetujui,None,028/LEAP/ST/IX/2023,,https://docs.google.com/document/d/1wMOD-N6oMQ...,https://docs.google.com/document/d/1xMv7AK2L9S...,done,,,2023-09-19 13:54:26
2,22,U00016,TEDxSurabaya Translators:\r\n\r\n1. Simulasi C...,TEDxSurabaya,NaT,Topic: Connect to TEDxSurabaya Join Zoom Meeti...,Online,Disetujui,None,025/LEAP/ST/VIII/2023,,https://docs.google.com/document/d/10Zi6g0R9RR...,https://docs.google.com/document/d/1pa9G3E3eUa...,laporan kegiatan done,,,2023-09-13 13:01:52
3,20,U00015,Pertemuan rutin di bulan Juli 2023 forum Laras...,Larasdikdudi,NaT,SMK Teknik PAL Surabaya Jalan Ujung Surabaya,Offline,Disetujui,None,023/LEAP/ST/VII/2023,,https://docs.google.com/document/d/1rILu6FE8Bd...,https://docs.google.com/document/d/1IAhWT4XjbX...,done/laporan sudah diisi,,,2023-07-25 09:56:19
4,21,U00015,Temu Warga RT 01 RW 08,Pengurus RT 01/ RW08,NaT,Balai RW,Offline,Disetujui,None,024/LEAP/ST/VIII/2023,,https://docs.google.com/document/d/1OQxpdKYfDF...,https://docs.google.com/document/d/1XrvRmYckBw...,,,,2023-08-22 17:43:48


Tipe data:
id_st                          int64
id_user                       object
acara                         object
undangan                      object
waktu_acara           datetime64[ns]
lokasi_acara                  object
jenis_kegiatan                object
status_st                     object
periode                       object
nomor_st                      object
catatan_st                    object
link_st                       object
link_laporan                  object
catatan_laporan               object
keterangan_st                 object
catatan_pembatalan            object
created_at            datetime64[ns]
dtype: object
Missing values (periode): 139


In [25]:
print("Contoh isi kolom waktu:")
print(df_st_raw['waktu'].head(10))
print("\nJumlah missing:", df_st_raw['waktu'].isnull().sum())

Contoh isi kolom waktu:
0    <table class="NormalTable">\r\n<tbody>\r\n<tr>...
1    <p><span class="fontstyle0">Hari, Tanggal : Ra...
2    <p>1. Simulasi Coaching:&nbsp;15 September 202...
3    <p><span class="fontstyle0">Hari, Tanggal : Ka...
4    <p>Rabu, 23 Agustus 2023<br />jam 19.00-selesa...
5    <p>Jum'at 8 September 2023</p>\r\n<p>Pukul 14....
6    <p>Minggu, 8 Oktober 2023</p>\r\n<p>Jam 17.15-...
7    <p>Jum'at, 22 September 2023 jam 15.00-16.30 W...
8    <p>hari, tanggal : Selasa, 26 September 2023</...
9    <p>hari, tanggal : Selasa, 26 September 2023</...
Name: waktu, dtype: object

Jumlah missing: 0


In [26]:
import re

# Mapping bulan Indonesia (jika belum ada)
bulan_map = {
    'januari': 1, 'februari': 2, 'maret': 3,
    'april': 4, 'mei': 5, 'juni': 6,
    'juli': 7, 'agustus': 8, 'september': 9,
    'oktober': 10, 'november': 11, 'desember': 12
}

def extract_date_id(text):
    if pd.isna(text) or not isinstance(text, str):
        return None
    pattern = r'(\d{1,2})\s+(Januari|Februari|Maret|April|Mei|Juni|Juli|Agustus|September|Oktober|November|Desember),?\s+(\d{4})'
    match = re.search(pattern, text, re.IGNORECASE)
    if match:
        hari = int(match.group(1))
        bulan = bulan_map[match.group(2).lower()]
        tahun = int(match.group(3))
        try:
            return pd.Timestamp(year=tahun, month=bulan, day=hari)
        except:
            return None
    return None

# Ekstrak tanggal dari kolom 'waktu'
df_st_raw['waktu_dt'] = df_st_raw['waktu'].apply(extract_date_id)

# Fallback: gunakan created_at untuk yang masih kosong
mask_null = df_st_raw['waktu_dt'].isnull()
print(f"Fallback untuk {mask_null.sum()} baris (pakai created_at).")
df_st_raw.loc[mask_null, 'waktu_dt'] = pd.to_datetime(df_st_raw.loc[mask_null, 'created_at'])

# Sekarang bangun DataFrame final surat_tugas
df_st = pd.DataFrame({
    'id_st': df_st_raw['idsurat'],
    'id_user': df_st_raw['idusers'].str.strip(),
    'acara': df_st_raw['acara'].str.strip(),
    'undangan': df_st_raw['undangan'].str.strip(),
    'waktu_acara': df_st_raw['waktu_dt'],
    'lokasi_acara': df_st_raw['lokasi'].str.strip(),
    'jenis_kegiatan': df_st_raw['jenis'].str.strip(),
    'status_st': df_st_raw['status'].str.strip(),
    'periode': None,
    'nomor_st': df_st_raw['nosurat'].fillna('').str.strip(),
    'catatan_st': df_st_raw['catatan'].fillna('').str.strip(),
    'link_st': df_st_raw['link'].fillna('').str.strip(),
    'link_laporan': df_st_raw['linklaporan'].fillna('').str.strip(),
    'catatan_laporan': df_st_raw['notelaporan'].fillna('').str.strip(),
    'keterangan_st': df_st_raw['ket'].fillna('').str.strip(),
    'catatan_pembatalan': df_st_raw['notebatal'].fillna('').str.strip(),
    'created_at': pd.to_datetime(df_st_raw['created_at'])
})


print("✅ df_st siap:")
display(df_st.head())
print("Tipe data:")
print(df_st.dtypes)
print("Missing values:")
print(df_st.isnull().sum())

# Simpan mapping id_st (lama -> baru)
mapping_st = dict(zip(df_st_raw['idsurat'], df_st['id_st']))
print("Mapping surat_tugas (lama -> baru):", list(mapping_st.items())[:5])

Fallback untuk 14 baris (pakai created_at).
✅ df_st siap:


,id_st,id_user,acara,undangan,waktu_acara,lokasi_acara,jenis_kegiatan,status_st,periode,nomor_st,catatan_st,link_st,link_laporan,catatan_laporan,keterangan_st,catatan_pembatalan,created_at
0,24,U00015,Sosialisasi Kerjasama LKP dan PKBM dengan Seme...,Dinas Pendidikan Kota Surabaya (bu Hilda),2023-09-20,"Tempat : Ruang Bung Tomo, Dinas Pendidikan Kot...",Offline,Disetujui,None,027/LEAP/ST/IX/2023,,https://docs.google.com/document/d/1LBx4WpqXcY...,https://docs.google.com/document/d/1jJdGMkonp_...,,,,2023-09-19 13:55:50
1,23,U00015,Pertemuan Rutin Larasdikdudi bulan September 2023,https://drive.google.com/drive/u/3/folders/1OI...,2023-09-27,AULA SMK Negeri 2 Surabaya Jalan Tentara Genie...,Offline,Disetujui,None,028/LEAP/ST/IX/2023,,https://docs.google.com/document/d/1wMOD-N6oMQ...,https://docs.google.com/document/d/1xMv7AK2L9S...,done,,,2023-09-19 13:54:26
2,22,U00016,TEDxSurabaya Translators:\r\n\r\n1. Simulasi C...,TEDxSurabaya,2023-09-15,Topic: Connect to TEDxSurabaya Join Zoom Meeti...,Online,Disetujui,None,025/LEAP/ST/VIII/2023,,https://docs.google.com/document/d/10Zi6g0R9RR...,https://docs.google.com/document/d/1pa9G3E3eUa...,laporan kegiatan done,,,2023-09-13 13:01:52
3,20,U00015,Pertemuan rutin di bulan Juli 2023 forum Laras...,Larasdikdudi,2023-07-27,SMK Teknik PAL Surabaya Jalan Ujung Surabaya,Offline,Disetujui,None,023/LEAP/ST/VII/2023,,https://docs.google.com/document/d/1rILu6FE8Bd...,https://docs.google.com/document/d/1IAhWT4XjbX...,done/laporan sudah diisi,,,2023-07-25 09:56:19
4,21,U00015,Temu Warga RT 01 RW 08,Pengurus RT 01/ RW08,2023-08-23,Balai RW,Offline,Disetujui,None,024/LEAP/ST/VIII/2023,,https://docs.google.com/document/d/1OQxpdKYfDF...,https://docs.google.com/document/d/1XrvRmYckBw...,,,,2023-08-22 17:43:48


Tipe data:
id_st                          int64
id_user                       object
acara                         object
undangan                      object
waktu_acara           datetime64[ns]
lokasi_acara                  object
jenis_kegiatan                object
status_st                     object
periode                       object
nomor_st                      object
catatan_st                    object
link_st                       object
link_laporan                  object
catatan_laporan               object
keterangan_st                 object
catatan_pembatalan            object
created_at            datetime64[ns]
dtype: object
Missing values:
id_st                   0
id_user                 0
acara                   0
undangan                0
waktu_acara             0
lokasi_acara            0
jenis_kegiatan          0
status_st               0
periode               139
nomor_st                0
catatan_st              0
link_st                 0
link_laporan      

In [27]:
# =========================================================
# ALTER TABEL SURAT_TUGAS: perbesar kolom lokasi_acara
# =========================================================
print("Mengubah tipe kolom lokasi_acara menjadi VARCHAR(500)...")

# 1. Cek tipe saat ini
current_type = pd.read_sql("""
    SELECT DATA_TYPE, CHARACTER_MAXIMUM_LENGTH 
    FROM INFORMATION_SCHEMA.COLUMNS 
    WHERE TABLE_NAME = 'surat_tugas' AND COLUMN_NAME = 'lokasi_acara'
""", db_future)
print("Tipe saat ini:", current_type)

# 2. Jalankan ALTER TABLE
try:
    with db_future.cursor() as cursor:
        cursor.execute("ALTER TABLE surat_tugas MODIFY COLUMN lokasi_acara VARCHAR(500) NOT NULL")
        db_future.commit()
    print("✅ Kolom lokasi_acara berhasil diubah menjadi VARCHAR(500)")
except Exception as e:
    print(f"❌ Error: {e}")

# 3. Verifikasi perubahan
new_type = pd.read_sql("""
    SELECT DATA_TYPE, CHARACTER_MAXIMUM_LENGTH 
    FROM INFORMATION_SCHEMA.COLUMNS 
    WHERE TABLE_NAME = 'surat_tugas' AND COLUMN_NAME = 'lokasi_acara'
""", db_future)
print("Tipe baru:", new_type)

Mengubah tipe kolom lokasi_acara menjadi VARCHAR(500)...
Tipe saat ini:   DATA_TYPE  CHARACTER_MAXIMUM_LENGTH
0   varchar                       255
1   varchar                       255
2   varchar                       500
3   varchar                       500
4   varchar                       255
5   varchar                       255
6   varchar                       255
7   varchar                       255
8   varchar                       255
✅ Kolom lokasi_acara berhasil diubah menjadi VARCHAR(500)
Tipe baru:   DATA_TYPE  CHARACTER_MAXIMUM_LENGTH
0   varchar                       255
1   varchar                       255
2   varchar                       500
3   varchar                       500
4   varchar                       255
5   varchar                       255
6   varchar                       255
7   varchar                       500
8   varchar                       255


In [28]:
# =========================================================
# CLEANING KHUSUS UNTUK SURAT TUGAS (df_st)
# =========================================================
print("Membersihkan df_st...")

# 1. Cek panjang lokasi_acara dan link_st > 500
if df_st['lokasi_acara'].str.len().max() > 500:
    print(f"⚠️ Ada lokasi_acara melebihi 500: {df_st[df_st['lokasi_acara'].str.len() > 500]['lokasi_acara'].tolist()}")
else:
    print("✓ lokasi_acara aman, max length:", df_st['lokasi_acara'].str.len().max())

if df_st['link_st'].str.len().max() > 255:
    print(f"⚠️ Ada link_st melebihi 255: {df_st[df_st['link_st'].str.len() > 255]['link_st'].tolist()}")
else:
    print("✓ link_st aman, max length:", df_st['link_st'].str.len().max())

# 2. Isi default untuk kolom yang null atau tanda '-'
df_st['undangan'] = df_st['undangan'].replace("", 'Undangan belum tersedia')
df_st['undangan'] = df_st['undangan'].replace('-', 'Undangan belum tersedia')

df_st['lokasi_acara'] = df_st['lokasi_acara'].replace("", 'Lokasi belum tersedia')
df_st['lokasi_acara'] = df_st['lokasi_acara'].replace('-', 'Lokasi belum tersedia')

df_st['catatan_st'] = df_st['catatan_st'].replace("", 'Tidak ada catatan')
df_st['catatan_st'] = df_st['catatan_st'].replace('-', 'Tidak ada catatan')

df_st['link_laporan'] = df_st['link_laporan'].replace("", 'Link laporan belum tersedia')
df_st['link_laporan'] = df_st['link_laporan'].replace('-', 'Link laporan belum tersedia')

df_st['catatan_laporan'] = df_st['catatan_laporan'].replace("", 'Tidak ada catatan')
df_st['catatan_laporan'] = df_st['catatan_laporan'].replace('-', 'Tidak ada catatan')

df_st['catatan_pembatalan'] = df_st['catatan_pembatalan'].replace("", 'Tidak ada catatan')

# 3. Replace nomor_st untuk id_st tertentu
df_st.loc[df_st['id_st'] == 83, 'nomor_st'] = '001/HR/ST/MIM/II/2024'
df_st.loc[df_st['id_st'] == 79, 'nomor_st'] = 'Nomor belum tersedia'

print("✓ Cleaning selesai")

Membersihkan df_st...
✓ lokasi_acara aman, max length: 270
✓ link_st aman, max length: 131
✓ Cleaning selesai


In [29]:
display(df_st.head())

,id_st,id_user,acara,undangan,waktu_acara,lokasi_acara,jenis_kegiatan,status_st,periode,nomor_st,catatan_st,link_st,link_laporan,catatan_laporan,keterangan_st,catatan_pembatalan,created_at
0,24,U00015,Sosialisasi Kerjasama LKP dan PKBM dengan Seme...,Dinas Pendidikan Kota Surabaya (bu Hilda),2023-09-20,"Tempat : Ruang Bung Tomo, Dinas Pendidikan Kot...",Offline,Disetujui,None,027/LEAP/ST/IX/2023,Tidak ada catatan,https://docs.google.com/document/d/1LBx4WpqXcY...,https://docs.google.com/document/d/1jJdGMkonp_...,Tidak ada catatan,,Tidak ada catatan,2023-09-19 13:55:50
1,23,U00015,Pertemuan Rutin Larasdikdudi bulan September 2023,https://drive.google.com/drive/u/3/folders/1OI...,2023-09-27,AULA SMK Negeri 2 Surabaya Jalan Tentara Genie...,Offline,Disetujui,None,028/LEAP/ST/IX/2023,Tidak ada catatan,https://docs.google.com/document/d/1wMOD-N6oMQ...,https://docs.google.com/document/d/1xMv7AK2L9S...,done,,Tidak ada catatan,2023-09-19 13:54:26
2,22,U00016,TEDxSurabaya Translators:\r\n\r\n1. Simulasi C...,TEDxSurabaya,2023-09-15,Topic: Connect to TEDxSurabaya Join Zoom Meeti...,Online,Disetujui,None,025/LEAP/ST/VIII/2023,Tidak ada catatan,https://docs.google.com/document/d/10Zi6g0R9RR...,https://docs.google.com/document/d/1pa9G3E3eUa...,laporan kegiatan done,,Tidak ada catatan,2023-09-13 13:01:52
3,20,U00015,Pertemuan rutin di bulan Juli 2023 forum Laras...,Larasdikdudi,2023-07-27,SMK Teknik PAL Surabaya Jalan Ujung Surabaya,Offline,Disetujui,None,023/LEAP/ST/VII/2023,Tidak ada catatan,https://docs.google.com/document/d/1rILu6FE8Bd...,https://docs.google.com/document/d/1IAhWT4XjbX...,done/laporan sudah diisi,,Tidak ada catatan,2023-07-25 09:56:19
4,21,U00015,Temu Warga RT 01 RW 08,Pengurus RT 01/ RW08,2023-08-23,Balai RW,Offline,Disetujui,None,024/LEAP/ST/VIII/2023,Tidak ada catatan,https://docs.google.com/document/d/1OQxpdKYfDF...,https://docs.google.com/document/d/1XrvRmYckBw...,Tidak ada catatan,,Tidak ada catatan,2023-08-22 17:43:48


In [30]:
df_stu_raw = pd.read_sql("SELECT * FROM surattugas_users", db_old)
print("=== SURAT TUGAS USERS (lama) ===")
print("Jumlah:", len(df_stu_raw))
display(df_stu_raw.head(10))
print("\nKolom:", df_stu_raw.columns.tolist())
print("\nTipe data:")
print(df_stu_raw.dtypes)
print("\nMissing values:")
print(df_stu_raw.isnull().sum())

=== SURAT TUGAS USERS (lama) ===
Jumlah: 319


,idsu,idsurat,idusers
0,15,6,U00012
1,16,6,U00003
2,17,7,U00026
3,18,7,U00012
4,19,8,U00026
5,20,8,U00018
6,21,9,U00026
7,22,9,U00016
8,23,9,U00018
9,24,10,U00011



Kolom: ['idsu', 'idsurat', 'idusers']

Tipe data:
idsu        int64
idsurat     int64
idusers    object
dtype: object

Missing values:
idsu       0
idsurat    0
idusers    0
dtype: int64


In [31]:
# Bentuk DataFrame surat_tugas_anggota
df_sta = pd.DataFrame({
    'id_st': df_stu_raw['idsurat'],       # sama dengan id_st di surat_tugas
    'id_user': df_stu_raw['idusers'].str.strip()
})

print("✅ surat_tugas_anggota siap:")
display(df_sta.head())
print("Tipe data:")
print(df_sta.dtypes)
print("Missing values:")
print(df_sta.isnull().sum())


✅ surat_tugas_anggota siap:


,id_st,id_user
0,6,U00012
1,6,U00003
2,7,U00026
3,7,U00012
4,8,U00026


Tipe data:
id_st       int64
id_user    object
dtype: object
Missing values:
id_st      0
id_user    0
dtype: int64


In [32]:
invalid_id = df_sta[
    ~df_sta['id_st'].isin(df_sta['id_st'])
]

print(invalid_id)

Empty DataFrame
Columns: [id_st, id_user]
Index: []


In [33]:
print(invalid_id['id_st'].unique())

[]


In [34]:
df_sta = df_sta[
    df_sta['id_st'].isin(df_sta['id_st'])
]

## SOP dan SOP Kategori

In [35]:
# Ambil data sopkategori dari DB lama
df_kat_raw = pd.read_sql("SELECT * FROM sopkategori", db_old)
print("Data sopkategori lama:")
display(df_kat_raw.head())

# Buat ID integer urut (1,2,3,...) untuk id_sop_kategori baru
old_ids = df_kat_raw['idsopkategori'].tolist()
mapping_kat = {old: i+1 for i, old in enumerate(old_ids)}

print("Mapping kategori (lama -> baru):")
for k, v in mapping_kat.items():
    print(f"  {k} -> {v}")

# Buat DataFrame sop_kategori
df_sop_kategori = pd.DataFrame({
    'id_sop_kategori': range(1, len(df_kat_raw) + 1),
    'nama_kategori_sop': df_kat_raw['nama'].fillna('').str.strip()
})

# =========================================================
# HAPUS DATA TRIAL PADA DF_SOP_KATEGORI
# =========================================================
print("Menghapus id_sop_kategori = 3 (data trial)...")

# Tampilkan sebelum hapus
print("Sebelum:", len(df_sop_kategori))
display(df_sop_kategori)

# Filter: hapus id_sop_kategori == 3
df_sop_kategori = df_sop_kategori[df_sop_kategori['id_sop_kategori'] != 3]

# Tampilkan setelah hapus
print("Sesudah:", len(df_sop_kategori))
display(df_sop_kategori)

# Simpan ke dictionary fase3
fase3_data = {}
fase3_data['sop_kategori'] = df_sop_kategori
print("✅ sop_kategori selesai")

Data sopkategori lama:


,idsopkategori,nama
0,K00001,Kelas
1,K00002,HR / GA
2,K00003,test


Mapping kategori (lama -> baru):
  K00001 -> 1
  K00002 -> 2
  K00003 -> 3
Menghapus id_sop_kategori = 3 (data trial)...
Sebelum: 3


,id_sop_kategori,nama_kategori_sop
0,1,Kelas
1,2,HR / GA
2,3,test


Sesudah: 2


,id_sop_kategori,nama_kategori_sop
0,1,Kelas
1,2,HR / GA


✅ sop_kategori selesai


In [36]:
# =================================================
# PROSES TABEL: sop (tanpa id_sop)
# =================================================

# 1. Ambil data mentah dari DB lama
df_sop_raw = pd.read_sql("SELECT * FROM sop", db_old)
print("Data SOP mentah:")
display(df_sop_raw.head())

# 2. Mapping kolom, bersihkan, dan ubah tipe data
df_sop = pd.DataFrame({
    'judul_sop':           df_sop_raw['judulsop'].fillna('').str.strip(),
    'link_dokumen_sop':    df_sop_raw['link'].fillna('').str.strip(),
    'created_at':          pd.to_datetime(df_sop_raw['created_at']),
    'id_sop_kategori':     df_sop_raw['idsopkategori'].map(mapping_kat)   # FK ke sop_kategori
})

# 3. Cek null & duplikat (meski tidak ada PK, kita lihat duplikat baris)
print("\nMissing values:")
print(df_sop.isnull().sum())
print(f"\nJumlah baris duplikat: {df_sop.duplicated().sum()}")

# 4. Simpan ke dictionary fase3
fase3_data['sop'] = df_sop

print("\n✅ DataFrame sop final (tanpa id_sop):")
display(df_sop.head())
print("\nTipe data:")
print(df_sop.dtypes)

Data SOP mentah:


,idsop,judulsop,link,created_at,idsopkategori
0,s00001,Akhir Penggunaan Kelas English (19.15 WIB),https://drive.google.com/file/d/1V0MpB7ctzPHe6...,2023-07-07 08:52:48,K00001
1,s00002,Penerimaan Surat Masuk,https://drive.google.com/file/d/1G8lO33yU7MufC...,2023-11-22 15:27:06,K00002
2,s00003,Pengajuan Surat Keluar,https://drive.google.com/file/d/14I7wYSk62WJAv...,2023-11-22 15:27:55,K00002
3,s00004,Pengajuan Surat Tugas,https://drive.google.com/file/d/1R-wKFqQeVSz62...,2023-11-22 15:28:27,K00002



Missing values:
judul_sop           0
link_dokumen_sop    0
created_at          0
id_sop_kategori     0
dtype: int64

Jumlah baris duplikat: 0

✅ DataFrame sop final (tanpa id_sop):


,judul_sop,link_dokumen_sop,created_at,id_sop_kategori
0,Akhir Penggunaan Kelas English (19.15 WIB),https://drive.google.com/file/d/1V0MpB7ctzPHe6...,2023-07-07 08:52:48,1
1,Penerimaan Surat Masuk,https://drive.google.com/file/d/1G8lO33yU7MufC...,2023-11-22 15:27:06,2
2,Pengajuan Surat Keluar,https://drive.google.com/file/d/14I7wYSk62WJAv...,2023-11-22 15:27:55,2
3,Pengajuan Surat Tugas,https://drive.google.com/file/d/1R-wKFqQeVSz62...,2023-11-22 15:28:27,2



Tipe data:
judul_sop                   object
link_dokumen_sop            object
created_at          datetime64[ns]
id_sop_kategori              int64
dtype: object


## 3. Transform Data (jika diperlukan)

In [37]:
import pickle

# Gabungkan semua DataFrame final yang sudah jadi
fase3_data = {
    "sop_kategori": df_sop_kategori,
    "sop": df_sop,
    "surat_keluar": df_sk,
    "verifikasi_surat_keluar": df_verif_final,
    "surat_tugas": df_st,
    "surat_tugas_anggota": df_sta,   # tanpa id_sop
}

# Simpan ke file pickle
with open("fase_3_afrida.pkl", "wb") as f:
    pickle.dump(fase3_data, f)

print("✅ fase_3_afrida.pkl berhasil disimpan!")
print("📦 Isi:")
for nama, df in fase3_data.items():
    print(f"   - {nama}: {len(df)} baris, kolom: {list(df.columns)}")

✅ fase_3_afrida.pkl berhasil disimpan!
📦 Isi:
   - sop_kategori: 2 baris, kolom: ['id_sop_kategori', 'nama_kategori_sop']
   - sop: 4 baris, kolom: ['judul_sop', 'link_dokumen_sop', 'created_at', 'id_sop_kategori']
   - surat_keluar: 231 baris, kolom: ['id_sk', 'id_user', 'keterangan_sk', 'link_dokumen_sk', 'status_sk', 'nomor_sk', 'catatan_sk', 'created_at']
   - verifikasi_surat_keluar: 513 baris, kolom: ['id_sk', 'status_verifikasi_sk', 'catatan_verifikasi_sk', 'created_at']
   - surat_tugas: 139 baris, kolom: ['id_st', 'id_user', 'acara', 'undangan', 'waktu_acara', 'lokasi_acara', 'jenis_kegiatan', 'status_st', 'periode', 'nomor_st', 'catatan_st', 'link_st', 'link_laporan', 'catatan_laporan', 'keterangan_st', 'catatan_pembatalan', 'created_at']
   - surat_tugas_anggota: 319 baris, kolom: ['id_st', 'id_user']


In [38]:
from IPython.display import display

for key, df in fase3_data.items():
    print(f"\n📊 {key}")
    display(df.head())


📊 sop_kategori


,id_sop_kategori,nama_kategori_sop
0,1,Kelas
1,2,HR / GA



📊 sop


,judul_sop,link_dokumen_sop,created_at,id_sop_kategori
0,Akhir Penggunaan Kelas English (19.15 WIB),https://drive.google.com/file/d/1V0MpB7ctzPHe6...,2023-07-07 08:52:48,1
1,Penerimaan Surat Masuk,https://drive.google.com/file/d/1G8lO33yU7MufC...,2023-11-22 15:27:06,2
2,Pengajuan Surat Keluar,https://drive.google.com/file/d/14I7wYSk62WJAv...,2023-11-22 15:27:55,2
3,Pengajuan Surat Tugas,https://drive.google.com/file/d/1R-wKFqQeVSz62...,2023-11-22 15:28:27,2



📊 surat_keluar


...

,id_sk,status_verifikasi_sk,catatan_verifikasi_sk,created_at
0,<NA>,Diajukan,Tidak ada catatan,2023-06-12 13:32:50
1,<NA>,Diajukan,Tidak ada catatan,2023-06-12 13:33:07
2,<NA>,Diajukan,Tidak ada catatan,2023-06-12 14:28:56
3,<NA>,Diajukan,Tidak ada catatan,2023-06-30 15:30:35
4,<NA>,Diajukan,Tidak ada catatan,2023-07-01 20:35:43



📊 surat_tugas


,id_st,id_user,acara,undangan,waktu_acara,lokasi_acara,jenis_kegiatan,status_st,periode,nomor_st,catatan_st,link_st,link_laporan,catatan_laporan,keterangan_st,catatan_pembatalan,created_at
0,24,U00015,Sosialisasi Kerjasama LKP dan PKBM dengan Seme...,Dinas Pendidikan Kota Surabaya (bu Hilda),2023-09-20,"Tempat : Ruang Bung Tomo, Dinas Pendidikan Kot...",Offline,Disetujui,None,027/LEAP/ST/IX/2023,Tidak ada catatan,https://docs.google.com/document/d/1LBx4WpqXcY...,https://docs.google.com/document/d/1jJdGMkonp_...,Tidak ada catatan,,Tidak ada catatan,2023-09-19 13:55:50
1,23,U00015,Pertemuan Rutin Larasdikdudi bulan September 2023,https://drive.google.com/drive/u/3/folders/1OI...,2023-09-27,AULA SMK Negeri 2 Surabaya Jalan Tentara Genie...,Offline,Disetujui,None,028/LEAP/ST/IX/2023,Tidak ada catatan,https://docs.google.com/document/d/1wMOD-N6oMQ...,https://docs.google.com/document/d/1xMv7AK2L9S...,done,,Tidak ada catatan,2023-09-19 13:54:26
2,22,U00016,TEDxSurabaya Translators:\r\n\r\n1. Simulasi C...,TEDxSurabaya,2023-09-15,Topic: Connect to TEDxSurabaya Join Zoom Meeti...,Online,Disetujui,None,025/LEAP/ST/VIII/2023,Tidak ada catatan,https://docs.google.com/document/d/10Zi6g0R9RR...,https://docs.google.com/document/d/1pa9G3E3eUa...,laporan kegiatan done,,Tidak ada catatan,2023-09-13 13:01:52
3,20,U00015,Pertemuan rutin di bulan Juli 2023 forum Laras...,Larasdikdudi,2023-07-27,SMK Teknik PAL Surabaya Jalan Ujung Surabaya,Offline,Disetujui,None,023/LEAP/ST/VII/2023,Tidak ada catatan,https://docs.google.com/document/d/1rILu6FE8Bd...,https://docs.google.com/document/d/1IAhWT4XjbX...,done/laporan sudah diisi,,Tidak ada catatan,2023-07-25 09:56:19
4,21,U00015,Temu Warga RT 01 RW 08,Pengurus RT 01/ RW08,2023-08-23,Balai RW,Offline,Disetujui,None,024/LEAP/ST/VIII/2023,Tidak ada catatan,https://docs.google.com/document/d/1OQxpdKYfDF...,https://docs.google.com/document/d/1XrvRmYckBw...,Tidak ada catatan,,Tidak ada catatan,2023-08-22 17:43:48



📊 surat_tugas_anggota


,id_st,id_user
0,6,U00012
1,6,U00003
2,7,U00026
3,7,U00012
4,8,U00026


## 4. Insert ke DB Baru

## 5. Verifikasi Data

## 6. Return Hasil Migrasi untuk migrate_db.py

## 7. Close Connection